In [ ]:
import os
import sys

%load_ext autoreload
%autoreload 2
import logging
# Configure logging
logging.basicConfig(level=logging.INFO)

# If your current working directory is the notebooks directory, use this:
notebook_dir = os.getcwd()  # current working directory
src_path = os.path.abspath(os.path.join(notebook_dir, '..', 'src'))
sys.path.append(src_path)

# Add the parent directory to sys.path
parent_dir = os.path.abspath(os.path.join(notebook_dir, '..'))
sys.path.append(parent_dir)
import pickle
from server_config import datapath, preprocessed_path_freezed, redcap_path, preprocessed_path


In [ ]:
import pandas as pd
import numpy as np


In [ ]:
from functions.preprocessing import gps_features
from functions.preprocessing.ema_mappings import run_ema_mappings
from functions.preprocessing.missing_data import summarize_missing_data

In [ ]:
import warnings
# Suppress only SettingWithCopyWarning
warnings.simplefilter(action='ignore', category=pd.errors.SettingWithCopyWarning)

In [ ]:
backup_path = preprocessed_path_freezed + "/backup_data_passive_actual.feather"
df_backup = pd.read_feather(backup_path)

In [ ]:
df_backup.type.unique()

In [ ]:
df_backup["delta_t"] = (df_backup["endTimestamp"] - df_backup["startTimestamp"]).dt.total_seconds()
delta_t = df_backup["delta_t"]

In [ ]:
# 1 sec min and 24h max, mostly 60sec
delta_t.describe()

In [ ]:
df_backup.groupby(["customer", "type"])

In [ ]:
df_backup.groupby("type")["customer"].nunique().sort_values(ascending=False).to_markdown()
# Convert the result to a DataFrame and print as markdown table
result = df_backup.groupby("type")["customer"].nunique().sort_values(ascending=False)
result_df = result.reset_index()
result_df.columns = ['Data Type', 'Unique Customers']
print(result_df.to_markdown(index=False))

In [ ]:
df_backup[df_backup["type"] == "ActiveBurnedCalories"].groupby(
    ["customer", "startTimestamp_day"]
)["doubleValue"].sum().sort_values(ascending=False)

In [ ]:
target_date = pd.to_datetime("2023-12-12")
df_backup[(df_backup["customer"] == "1BAf") & (df_backup["startTimestamp_day"] == target_date)]
# first record says that 17k calories were burned on that day something is off with this 1BAf customer

In [ ]:
df_backup[["SleepStateBinary","SleepBinary"]][df_backup["customer"] == df_backup["customer"][0]]

In [ ]:
df_backup.groupby("type")["delta_t"].describe()

In [ ]:
86399 / 24

In [ ]:
len(delta_t)

In [ ]:
(delta_t == 60).sum() / delta_t.count()